# Experiment 9: Data Cleaning and Preparation
## Problem: Analyzing Customer Churn in a Telecommunications Company

**Aim:** To clean and prepare the `Telecom_Customer_Churn.csv` dataset for customer churn analysis.

**Instructions:** Keep `Telecom_Customer_Churn.csv` in the same folder as this notebook, then run the cells from top to bottom.

## Step 1: Import libraries and dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Telecom_Customer_Churn.csv")
df.head()

## Step 2: Explore the dataset

In [ ]:
# Number of rows and columns
print("Dataset shape:", df.shape)

# Column names
print("\nColumn names:")
print(df.columns.tolist())

# Dataset information
print("\nDataset information:")
df.info()

In [ ]:
# Statistical summary
df.describe(include="all")

## Step 3: Handle missing values

In [ ]:
# Check missing values
df.isnull().sum()

In [ ]:
# Fill missing numerical values with the median
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with the most common value (mode)
cat_cols = df.select_dtypes(include=["object", "string"]).columns
for col in cat_cols:
    if df[col].notna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after cleaning:")
print(df.isnull().sum())

## Step 4: Remove duplicate records

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

## Step 5: Standardize inconsistent data

In [ ]:
# Remove extra spaces from text values
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip()

# Inspect category values for spelling or formatting differences
for col in df.select_dtypes(include=["object", "string"]).columns:
    print("\n", col, ":", df[col].unique()[:20])

## Step 6: Convert columns to the correct data types

In [ ]:
# Convert TotalCharges to numeric if this column exists
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Display data types
df.dtypes

In [ ]:
# If conversion created missing values, fill numeric columns with their median
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print(df.isnull().sum())

## Step 7: Identify and handle outliers

In [ ]:
# Check outliers in MonthlyCharges, if the column exists
if "MonthlyCharges" in df.columns:
    Q1 = df["MonthlyCharges"].quantile(0.25)
    Q3 = df["MonthlyCharges"].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df["MonthlyCharges"] < lower) | (df["MonthlyCharges"] > upper)]
    print("MonthlyCharges outliers:", len(outliers))

    # Cap values at the IQR limits to retain customer records
    df["MonthlyCharges"] = df["MonthlyCharges"].clip(lower, upper)
else:
    print("MonthlyCharges column not found; check df.columns.")

## Step 8: Feature engineering

In [ ]:
# Create tenure groups if tenure exists
if "tenure" in df.columns:
    df["TenureGroup"] = pd.cut(
        df["tenure"],
        bins=[-1, 12, 24, 48, float("inf")],
        labels=["New", "Medium", "Long", "Very Long"]
    )

# Create an indicator for above-median monthly charges
if "MonthlyCharges" in df.columns:
    median_charge = df["MonthlyCharges"].median()
    df["HighMonthlyCharge"] = (df["MonthlyCharges"] > median_charge).astype(int)

df.head()

## Step 9: Normalize numerical data

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Scale common continuous numeric columns when available
scale_cols = [
    col for col in ["tenure", "MonthlyCharges", "TotalCharges"]
    if col in df.columns
]

if scale_cols:
    scaler = MinMaxScaler()
    df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("Scaled columns:", scale_cols)
df.head()

## Step 10: Save the cleaned dataset

In [ ]:
df.to_csv("Cleaned_Telecom_Customer_Churn.csv", index=False)

print("Data cleaning and preparation completed successfully!")
print("Final dataset shape:", df.shape)
print("Saved as Cleaned_Telecom_Customer_Churn.csv")

## Result

The telecommunications customer churn dataset was explored and prepared by handling missing values, removing duplicate records, standardizing text, correcting data types, identifying and capping outliers, creating new features, and scaling selected numerical columns. The cleaned dataset was saved as `Cleaned_Telecom_Customer_Churn.csv`.

**Note:** The notebook expects a CSV named `Telecom_Customer_Churn.csv` in the same folder. Some steps check for common column names such as `MonthlyCharges`, `TotalCharges`, and `tenure`; if your dataset uses different names, update those cells accordingly.